# Macro stress test: unemployment shock → credit card losses

**Ask from Risk leadership:** if US unemployment rises ~2 points, what happens to losses on our card portfolio over the next 2 years?

_Data: FRED (public). Model: interpretable, illustrative — not a forecast._

## 1. Acquire
Find candidate series straight from FRED's own catalog (`fred/series/search`), then pick by frequency, seasonal adjustment and coverage.

In [ ]:
import os, requests, pandas as pd
from IPython.display import display

def fred_search(text, limit=5):
    r = requests.get("https://api.stlouisfed.org/fred/series/search",
                     params={"search_text": text, "api_key": os.environ["FRED_API_KEY"], "file_type": "json",
                             "order_by": "popularity", "sort_order": "desc", "limit": limit}, timeout=20)
    r.raise_for_status()
    cols = ["id", "title", "frequency_short", "seasonal_adjustment_short", "units", "observation_end"]
    return pd.DataFrame(r.json()["seriess"]).reindex(columns=cols)

try:
    for q in ["credit card delinquency rate commercial banks", "credit card charge-off rate",
              "unemployment rate", "effective federal funds rate", "consumer price index all urban"]:
        print(f"\n🔎 {q}"); display(fred_search(q))
except Exception as exc:
    print(f"FRED search unavailable ({type(exc).__name__}); using the vetted series IDs below.")

**Selected series**

| Column | FRED id | Frequency | Notes |
|---|---|---|---|
| chargeoff | CORCCACBS | Quarterly, SA | Credit card charge-off rate — realised losses |
| delinquency | DRCCLACBS | Quarterly, SA | Credit card delinquency rate |
| unrate | UNRATE | Monthly, SA | Civilian unemployment rate |
| fedfunds | FEDFUNDS | Monthly | Effective fed funds rate |
| cpi | CPIAUCSL | Monthly, SA | CPI-U, used for YoY inflation |

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED = ROOT / "data" / "seed"
SERIES = {"delinquency": "DRCCLACBS", "chargeoff": "CORCCACBS",
          "unrate": "UNRATE", "fedfunds": "FEDFUNDS", "cpi": "CPIAUCSL"}

def to_series(rows):
    # FRED encodes missing observations as "." -> NaN (never zero)
    df = pd.DataFrame(rows)
    return pd.Series(pd.to_numeric(df["value"].replace(".", np.nan), errors="coerce").values,
                     index=pd.to_datetime(df["date"]), name="value").sort_index()

def fetch(series_id, start="1990-01-01"):
    r = requests.get("https://api.stlouisfed.org/fred/series/observations",
                     params={"series_id": series_id, "api_key": os.environ["FRED_API_KEY"],
                             "file_type": "json", "observation_start": start}, timeout=20)
    r.raise_for_status()
    return to_series(r.json()["observations"])

try:
    raw = {name: fetch(sid) for name, sid in SERIES.items()}
    source = "live FRED API"
except Exception as exc:  # network/key problems -> cached seed data
    raw = {name: to_series(pd.read_csv(SEED / f"{sid}.csv", dtype={"value": str}).to_dict("records"))
           for name, sid in SERIES.items()}
    source = f"cached seed data ({type(exc).__name__})"

print("Source:", source)
pd.DataFrame({n: {"first": s.index.min().date(), "last": s.index.max().date(),
                  "obs": len(s), "missing": int(s.isna().sum())} for n, s in raw.items()}).T

## 2. Clean & align
Charge-offs and delinquency are quarterly; the macro series are monthly. Average the available months in each quarter (at least 2 of 3 — FRED has the odd missing month), keep the quarter sequence **gap-free** so lags never jump across a hole, and derive YoY CPI inflation.

In [ ]:
def quarterly(s, name):
    s = s.dropna()
    g = s.groupby(s.index.to_period("Q"))
    q = g.last() if name in ("delinquency", "chargeoff") else g.mean()[g.count() >= 2]
    q.index = q.index.to_timestamp(how="start")
    return q.rename(name)

panel = pd.concat([quarterly(s, n) for n, s in raw.items()], axis=1, sort=True).sort_index()
quarters = pd.period_range(panel.index.min().to_period("Q"), panel.index.max().to_period("Q"), freq="Q")
panel = panel.reindex(quarters.to_timestamp(how="start"))            # gap-free quarterly index
panel["cpi_yoy"] = panel["cpi"].pct_change(4, fill_method=None) * 100
panel = panel.drop(columns="cpi")
ok = panel[["chargeoff", "unrate"]].notna().all(axis=1)
panel = panel.loc[ok[ok].index.min(): ok[ok].index.max()]
panel.index.name = "date"
gaps = panel[panel[["chargeoff", "unrate"]].isna().any(axis=1)].index
print(f"{len(panel)} quarters: {panel.index.min().to_period('Q')} → {panel.index.max().to_period('Q')}; "
      f"interior gaps: {[str(g.to_period('Q')) for g in gaps] or 'none'}")
panel.tail()

## 3. Explore

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4), layout="constrained")
ax1.plot(panel.index, panel["unrate"], color="#4C72B0")
ax1.set_ylabel("Unemployment (%)", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(panel.index, panel["chargeoff"], color="#C44E52", label="Charge-offs")
ax2.plot(panel.index, panel["delinquency"], color="#DD8452", ls=":", label="Delinquency")
ax2.set_ylabel("Card loss metrics (%)"); ax2.legend(loc="upper left")
for lo, hi in [("2008-01-01", "2009-06-30"), ("2020-04-01", "2021-12-31")]:
    ax1.axvspan(pd.Timestamp(lo), pd.Timestamp(hi), color="grey", alpha=0.12)
ax1.set_title("Unemployment vs. credit card losses (shaded: GFC, pandemic)")
plt.show()

Losses rise when unemployment **rises**, then fall even while unemployment stays high (2010–14). And in 2020 unemployment spiked while losses *fell* — stimulus and forbearance.

### Aside: is the Phillips curve flatter than it used to be?
Rolling 10-year correlation between unemployment and inflation.

In [ ]:
roll = panel["unrate"].rolling(40, min_periods=36).corr(panel["cpi_yoy"])
fig, ax = plt.subplots(figsize=(11, 3), layout="constrained")
ax.plot(roll.index, roll.values); ax.set_title("Rolling 10y corr(unemployment, CPI inflation)")
ax.axhline(0, color="grey", lw=0.8); plt.show()

## 4. Model
### First attempt: delinquency on the *level* of unemployment

In [ ]:
import statsmodels.api as sm

def ols(y, X):
    d = pd.concat([y, X], axis=1, sort=True).dropna()
    return sm.OLS(d.iloc[:, 0], sm.add_constant(d.iloc[:, 1:])).fit(cov_type="HAC", cov_kwds={"maxlags": 4})

naive = ols(panel["delinquency"].loc["2000":],
            pd.DataFrame({"delinquency_lag1": panel["delinquency"].shift(1),
                          "unrate_lag1": panel["unrate"].shift(1),
                          "fedfunds_lag1": panel["fedfunds"].shift(1)}).loc["2000":])
print(naive.params.round(3).to_string())
print("\nSanity check — higher unemployment should mean higher losses:",
      "PASS" if naive.params["unrate_lag1"] > 0 else "FAIL (wrong sign)")

**It fails the sanity check.** The levels model can't explain 2010–14 (high but falling unemployment, falling losses) and is pulled further off by 2020. Fix:
- Target **charge-offs** (realised losses — what stress tests measure).
- Drive them with the **1-year change** in unemployment, lagged one quarter.
- **Exclude 2020Q2–2021Q4** from the fit (pandemic policy broke the link).

`chargeoff_t = a + b1·chargeoff_{t-1} + b2·(unrate_{t-1} − unrate_{t-5})`, OLS with HAC errors, 1991 onward.

In [ ]:
X = pd.DataFrame({"chargeoff_lag1": panel["chargeoff"].shift(1),
                  "unrate_chg4_lag1": (panel["unrate"] - panel["unrate"].shift(4)).shift(1)})
pandemic = (panel.index >= "2020-04-01") & (panel.index <= "2021-12-31")
keep = (panel.index >= "1991-01-01") & ~pandemic
res = ols(panel["chargeoff"][keep], X[keep])
with_covid = ols(panel["chargeoff"].loc["1991":], X.loc["1991":])

print(res.summary().tables[1])
b = res.params
print(f"R² = {res.rsquared:.3f}, n = {int(res.nobs)}, long-run mean = {b['const'] / (1 - b['chargeoff_lag1']):.2f}%")
print(f"Unemployment effect: {b['unrate_chg4_lag1']:+.2f} (t={res.tvalues['unrate_chg4_lag1']:.1f}) "
      f"vs {with_covid.params['unrate_chg4_lag1']:+.2f} if pandemic quarters are left in")
print("Sanity check:", "PASS" if b["unrate_chg4_lag1"] > 0 and 0 <= b["chargeoff_lag1"] < 1 else "FAIL")

## 5. Stress scenario: unemployment +2pp, ramped over 4 quarters
Assumptions: national data as a proxy for our book; relationships stable outside the pandemic; the baseline drifts toward the long-run mean, so read the **gap** between the lines.

In [ ]:
last = panel.dropna(subset=["chargeoff", "unrate"]).index.max()
u_hist = panel["unrate"].loc[:last].iloc[-5:].interpolate().tolist()   # last 5 quarters, gap-filled

def project(shock_pp, horizon=8, ramp=4):
    u, co, out = list(u_hist), panel.loc[last, "chargeoff"], []
    for h in range(1, horizon + 1):
        co = b["const"] + b["chargeoff_lag1"] * co + b["unrate_chg4_lag1"] * (u[-1] - u[-5])
        u.append(u_hist[-1] + shock_pp * min(h / ramp, 1))
        out.append(((last.to_period("Q") + h).to_timestamp(), u[-1], co))
    return pd.DataFrame(out, columns=["date", "unrate", "chargeoff"]).set_index("date")

base, shock = project(0.0), project(2.0)
hist = panel["chargeoff"].loc["2015":last]
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
ax.plot(hist.index, hist.values, color="black", label="History")
ax.plot(base.index, base["chargeoff"], ls="--", label="Baseline")
ax.plot(shock.index, shock["chargeoff"], color="crimson", label="+2pp unemployment")
ax.set_ylabel("Card charge-off rate (%)"); ax.legend(); ax.set_title("Illustrative stress path")
plt.show()
gap = shock["chargeoff"] - base["chargeoff"]
print(f"Peak stress impact: +{gap.max():.2f}pp charge-offs, {gap.idxmax().to_period('Q')} "
      f"({shock['chargeoff'].max():.2f}% vs {base['chargeoff'].max():.2f}% baseline)")

## 6. Graduated to production
The logic above now lives in the tested `risk_model` package. Publish the shared artifacts the dashboard and SRE checks consume (`docs/DATA_CONTRACT.md`).

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from risk_model import build_quarterly_panel, fit, sanity_check, stress_payload, storage
prod_panel = build_quarterly_panel(raw)
prod_model = fit(prod_panel)
assert abs(prod_model.coefficients["unrate_chg4_lag1"] - b["unrate_chg4_lag1"]) < 1e-9, "notebook and package disagree"
assert sanity_check(prod_model) == [], sanity_check(prod_model)
storage.publish(prod_panel, stress_payload(prod_panel, prod_model, shock_pp=2.0))